# Hands-on 31 (Bab 12): Membuat Alert Sederhana jika Skor Anomali Tinggi

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 12.

Sistem IoT-ML yang baik tidak berhenti pada prediksi; ia mendorong tindakan. Di sini
subscriber pipeline menerbitkan alert ke topik `iotlab/room01/alert` bila model
memprediksi FAULT, atau suhu > 32 C dan CO2 > 1200 ppm bersamaan. Alert juga disimpan
ke `outputs/reports/pipeline_room_alerts.csv` (InfluxDB opsional via `--influx`).

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")

# --- Utilitas InfluxDB (nilai bawaan skrip buku) ---
from influxdb_client import InfluxDBClient
INFLUX_URL, INFLUX_TOKEN = "http://localhost:8086", "GANTI_DENGAN_TOKEN_ANDA"
INFLUX_ORG, INFLUX_BUCKET = "iotlab", "sensor_data"
_ic = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
if not _ic.ping():
    raise RuntimeError("InfluxDB belum berjalan di localhost:8086 (org iotlab, bucket sensor_data, "
                       "token GANTI_DENGAN_TOKEN_ANDA). Skrip buku di notebook ini menulis ke InfluxDB.")

def influx_df(measurement, rentang="-30d"):
    import pandas as pd
    q = (f'from(bucket:"{INFLUX_BUCKET}") |> range(start: {rentang}, stop: 1d) '
         f'|> filter(fn:(r) => r._measurement == "{measurement}") '
         f'|> pivot(rowKey:["_time"], columnKey:["_field"], valueColumn:"_value")')
    res = _ic.query_api().query_data_frame(q)
    df = pd.concat(res) if isinstance(res, list) else res
    if len(df):
        df = df.drop(columns=[c for c in df.columns if c in ("result", "table", "_start", "_stop", "_measurement")])
        df = df.sort_values("_time").reset_index(drop=True)
    return df

def influx_hapus(measurement):
    _ic.delete_api().delete("1970-01-01T00:00:00Z", "2100-01-01T00:00:00Z",
                            f'_measurement="{measurement}"', bucket=INFLUX_BUCKET, org=INFLUX_ORG)


[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Jalankan Pipeline + Alert (broker, subscriber+alert, listener alert, simulator)

In [2]:
influx_hapus("room_pipeline"); influx_hapus("room_alerts")
broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = latar("scripts/pipeline_room_with_alert.py", "pipeline_alert_sub_nb.log")
    time.sleep(2)
    pub = latar("scripts/publisher_room_simulator.py", "pipeline_alert_pub_nb.log")
    time.sleep(40)   # sekitar 20 pesan
    henti(pub, sub)
    print("--- Output subscriber+alert ---")
    print(log_teks("pipeline_alert_sub_nb.log"))
finally:
    broker.terminate()

--- Output subscriber+alert ---
Subscriber+Alert terhubung ke broker.
Subscribe ke: iotlab/room01/telemetry
[2026-10-22T13:27:52] -> status=warning
[2026-10-22T13:27:54] -> status=normal
[2026-10-22T13:27:56] -> status=normal
[2026-10-22T13:27:58] -> status=normal
[2026-10-22T13:28:00] -> status=warning
[2026-10-22T13:28:02] -> status=warning
[2026-10-22T13:28:04] -> status=normal
[2026-10-22T13:28:06] -> status=warning
[2026-10-22T13:28:08] -> status=warning
[2026-10-22T13:28:10] -> status=normal
[2026-10-22T13:28:12] -> status=warning
[2026-10-22T13:28:14] -> status=warning
[2026-10-22T13:28:16] -> status=fault
ALERT: {'timestamp': '2026-10-22T13:28:16', 'device_id': 'room01', 'pred_status': 'fault', 'alert_reason': 'Model memprediksi FAULT', 'temperature_c': 29.99, 'humidity_pct': 42.2, 'co2_ppm': 1425, 'power_watt': 269.45}
[2026-10-22T13:28:18] -> status=warning
[2026-10-22T13:28:20] -> status=warning
ALERT: {'timestamp': '2026-10-22T13:28:20', 'device_id': 'room01', 'pred_status'

## 2. Verifikasi Log Alert

In [3]:
alerts = influx_df("room_alerts")
print(alerts.to_string(index=False) if len(alerts) else "Tidak ada alert pada batch ini.")

                    _time device_id pred_status  alert_flag  co2_ppm  humidity_pct  power_watt  temperature_c                  alert_reason
2026-10-22 13:27:52+00:00    room01     warning           1    596.0         58.55      133.96          29.81                           NaN
2026-10-22 13:27:54+00:00    room01      normal           1   1198.0         50.81       94.71          28.32                           NaN
2026-10-22 13:27:56+00:00    room01      normal           1    894.0         51.66      235.46          24.50                           NaN
2026-10-22 13:27:58+00:00    room01      normal           1   1177.0         52.88      176.94          27.88                           NaN
2026-10-22 13:28:00+00:00    room01     warning           1   1220.0         50.49      211.92          31.83                           NaN
2026-10-22 13:28:02+00:00    room01     warning           1    679.0         51.70      248.83          30.31                           NaN
2026-10-22 13:28:04+

## Penutup

Alert berbasis status model (FAULT) dan aturan sensor sederhana (suhu + CO2 tinggi)
diterbitkan ke topik terpisah dan tersimpan di storage. Pola ini mudah dikembangkan:
tambahkan ambang skor anomali, kirim ke notifikasi, atau tampilkan panel alert di
Grafana sesuai buku.

## Tambahan: Versi Sederhana Berbasis Skor Anomali

Pendekatan B buku: skor 0.4 (suhu > 31) + 0.4 (CO2 > 1100) + 0.2 (daya > 220), alert bila skor >= 0.7. Dijalankan pada dataset sintetis ruangan.

In [4]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/alert_score_demo.py"], capture_output=True, text=True)
print(r.stdout)

Total baris : 2160
Alert (skor >= 0.7): 18 (0.83%)
Distribusi skor:
0.0    2119
0.2       5
0.4       6
0.6      12
0.8       1
1.0      17

Contoh alert:
          timestamp device_id  temperature_c  humidity_pct  co2_ppm  power_watt status_label  is_anomaly
2026-01-01 07:10:00    room01      33.203553         47.17     1317  277.226732        fault           1
2026-01-02 17:40:00    room01      34.256756         51.99     1278  332.337377        fault           1
2026-01-03 02:20:00    room01      31.137017         51.87     1153  185.236149        fault           1
2026-01-04 09:40:00    room01      34.453892         45.75     1698  276.828258        fault           1
2026-01-04 17:40:00    room01      32.651549         45.13     1352  299.818694        fault           1



## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 33 dalam urutan buku) sebagai panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 9. Langkah kerja yang harus dilakukan pembaca

Hands-on 30
1. jalankan Mosquitto, InfluxDB, dan Grafana
2. jalankan simulator sensor
3. jalankan subscriber Python
4. validasi data masuk ke broker
5. pastikan preprocessing dan prediksi berjalan
6. periksa data dan hasil inferensi di InfluxDB
7. bangun dashboard Grafana

Hands-on 31
1. tambahkan logika alert ke subscriber
2. tentukan aturan alert sederhana
3. kirim alert ke topik MQTT khusus
4. simpan alert ke measurement khusus
5. tampilkan alert di dashboard
6. uji dengan nilai sensor ekstrem atau simulasi fault